# TALN — TP 3 : Convertir la parole / l’image en texte
**FST · 2026–2027 · Dr. Asma Mekki · IDS 4 / IGL 5**

Notebook correspondant aux deux exercices du sujet. Exécuter les cellules dans l’ordre. Les fichiers WAV ne sont pas fournis avec le PDF.

## Installation et configuration (Colab / Jupyter local)

Dans **Google Colab**, ne pas installer `pyaudio` : le microphone est accessible via le navigateur, pas via `sr.Microphone()`. Sous Linux/Colab, installer Tesseract et FFmpeg avec `apt`. Sur Windows local, installer Tesseract séparément et configurer son chemin si nécessaire. Les modèles EasyOCR seront téléchargés à la première exécution.

In [ ]:
%pip install -q SpeechRecognition pydub pytesseract easyocr jiwer language-tool-python pillow pandas matplotlib numpy

In [ ]:
# Installer les outils système seulement sur Linux (Colab compris)
import platform, shutil, subprocess
if platform.system() == "Linux":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "tesseract-ocr", "tesseract-ocr-eng", "tesseract-ocr-fra"], check=True)
else:
    print("Windows/macOS : vérifiez que FFmpeg et Tesseract sont installés et accessibles dans PATH.")
print("Tesseract :", shutil.which("tesseract"), "| FFmpeg :", shutil.which("ffmpeg"))

In [ ]:
from pathlib import Path
import os, time, re
import pandas as pd
from IPython.display import display
DATA = Path("data")
DATA.mkdir(exist_ok=True)
print("Placez exemple_1.wav et grand_audio.wav dans", DATA.resolve())

In [ ]:
# Optionnel : téléverser les fichiers WAV dans Google Colab
# Décommentez seulement si vous utilisez Colab :
# from google.colab import files
# for nom, contenu in files.upload().items():
#     (DATA / nom).write_bytes(contenu)
#     print("Ajouté :", DATA / nom)

## Exercice 1 — Reconnaissance vocale
### Questions 1 et 2 — Charger et transcrire `exemple_1.wav`
La reconnaissance Google de `SpeechRecognition` nécessite une connexion Internet.

In [ ]:
import speech_recognition as sr
r = sr.Recognizer()
audio_path = DATA / "exemple_1.wav"
if audio_path.exists():
    with sr.AudioFile(str(audio_path)) as source:
        audio = r.record(source)
    try:
        texte = r.recognize_google(audio, language="fr-FR")
        print("Transcription :", texte)
    except sr.UnknownValueError:
        print("Parole non reconnue.")
    except sr.RequestError as e:
        print("Service de reconnaissance indisponible :", e)
else:
    print("Fichier manquant :", audio_path, "— ajoutez-le pour tester.")

### Question 3 — Explication détaillée de `transcription_grand_audio()`
La fonction du sujet :

1. **`AudioSegment.from_wav(path)`** charge le fichier WAV.
2. **`split_on_silence`** découpe le signal aux silences : `min_silence_len=500` signifie au moins 500 ms de silence ; `silence_thresh=sound.dBFS-14` définit le seuil à 14 dB sous le niveau moyen du signal ; `keep_silence=500` conserve 500 ms de silence autour des segments.
3. **`os.makedirs`** crée le dossier `chunks` s’il n’existe pas.
4. La boucle **`enumerate(chunks, start=1)`** parcourt les segments numérotés.
5. **`export(..., format="wav")`** écrit chaque segment dans un fichier WAV.
6. **`sr.AudioFile` et `r.record`** chargent le segment pour la reconnaissance.
7. **`r.recognize_google`** transcrit chaque segment via le service Google.
8. **`sr.UnknownValueError`** gère un segment incompréhensible ; la version améliorée gère aussi `RequestError`.
9. **`capitalize()`** met la première lettre en majuscule et le reste en minuscules ; **`whole_text += text`** concatène les transcriptions ; **`return whole_text`** renvoie le résultat complet.

**Limites :** les pauses ne correspondent pas toujours aux phrases ; des segments peuvent être perdus ; la qualité dépend du bruit, de la langue et du service distant. Le code ci-dessous reprend les paramètres du sujet et ajoute quelques contrôles.

In [ ]:
from pydub import AudioSegment
from pydub.silence import split_on_silence

def transcription_grand_audio(path, langue="fr-FR", dossier="chunks"):
    sound = AudioSegment.from_wav(str(path))
    chunks = split_on_silence(sound, min_silence_len=500,
                              silence_thresh=sound.dBFS - 14, keep_silence=500)
    if not chunks:
        chunks = [sound]
    os.makedirs(dossier, exist_ok=True)
    whole_text = ""
    for i, audio_chunk in enumerate(chunks, start=1):
        chunk_filename = os.path.join(dossier, f"chunk{i}.wav")
        audio_chunk.export(chunk_filename, format="wav")
        with sr.AudioFile(chunk_filename) as source:
            audio_listened = r.record(source)
        try:
            text = r.recognize_google(audio_listened, language=langue)
        except sr.UnknownValueError:
            print(f"Segment {i} : parole non reconnue")
            continue
        except sr.RequestError as e:
            print(f"Segment {i} : service indisponible ({e})")
            continue
        text = f"{text.capitalize()}. "
        print(chunk_filename, ":", text)
        whole_text += text
    return whole_text.strip()

In [ ]:
grand_path = DATA / "grand_audio.wav"
if grand_path.exists():
    resultat_grand = transcription_grand_audio(grand_path, langue="fr-FR")
    print("\nTexte complet :", resultat_grand)
else:
    print("Ajoutez", grand_path, "pour effectuer ce test.")

### Question 4 — Microphone (5 secondes)

- **Colab** : utilisez la cellule d'enregistrement par navigateur ci-dessous ; autorisez l'accès au microphone.
- **Jupyter local** : `sr.Microphone()` exige PyAudio, à installer séparément uniquement sur l'ordinateur local.

La reconnaissance Google nécessite une connexion Internet.

In [ ]:
def transcription_microphone(langue="fr-FR", duree=5):
    recognizer = sr.Recognizer()
    try:
        with sr.Microphone() as source:
            print(f"Parlez pendant {duree} secondes ({langue})...")
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.record(source, duration=duree)
        resultat = recognizer.recognize_google(audio, language=langue)
        print("Texte reconnu :", resultat)
        return resultat
    except sr.UnknownValueError:
        print("Parole non reconnue.")
    except sr.RequestError as e:
        print("Erreur du service :", e)
    except (OSError, AttributeError, ImportError) as e:
        print("Microphone indisponible :", e)
    return None

# Décommentez pour enregistrer :
# transcription_microphone("fr-FR", duree=5)

In [ ]:
# Google Colab : enregistrement depuis le navigateur (5 secondes)
# À lancer dans Colab uniquement. Dans un Jupyter local, utilisez transcription_microphone().
def transcription_micro_colab(langue="fr-FR", duree=5):
    from google.colab import output
    from IPython.display import Javascript, display
    from base64 import b64decode
    import subprocess, tempfile
    js = r"""
    async function recordAudio(seconds) {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      const done = new Promise(resolve => recorder.onstop = resolve);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));
      recorder.stop();
      await done;
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(chunks, {type: recorder.mimeType});
      return await new Promise(resolve => {
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result.split(',')[1]);
        reader.readAsDataURL(blob);
      });
    }
    """
    display(Javascript(js))
    print(f"Autorisez le microphone et parlez pendant {duree} secondes.")
    encoded = output.eval_js(f"recordAudio({int(duree)})")
    with tempfile.TemporaryDirectory() as temp:
        raw = Path(temp) / "micro.webm"
        wav = Path(temp) / "micro.wav"
        raw.write_bytes(b64decode(encoded))
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-ar", "16000", "-ac", "1", str(wav)], check=True)
        with sr.AudioFile(str(wav)) as source:
            audio = sr.Recognizer().record(source)
        try:
            texte = sr.Recognizer().recognize_google(audio, language=langue)
            print("Texte reconnu :", texte)
            return texte
        except sr.UnknownValueError:
            print("Parole non reconnue.")
        except sr.RequestError as exc:
            print("Service indisponible :", exc)
    return None

# Décommentez dans Google Colab :
# transcription_micro_colab("fr-FR", duree=5)

### Question 5 — Fragments de test en anglais et français
Les phrases longues peuvent dépasser 5 secondes : augmenter `duree` au besoin. Chaque ligne de test déclenche un enregistrement distinct.

In [ ]:
phrases = {
 "en-US": ["Happy", "Sad", "Angry", "Natural Language Processing",
           "I believe Natural Language Processing is an interesting field",
           "Speech recognition is the ability of computer software to identify words and phrases in spoken language and convert them to human-readable text."],
 "fr-FR": ["Content", "Triste", "Fâché", "Traitement Automatique de la Langue Naturelle",
           "Je crois que le Traitement Automatique de la Langue Naturelle est un domaine intéressant",
           "La reconnaissance vocale est la capacité d’un logiciel informatique à identifier des mots et des phrases dans la langue parlée et à les convertir en texte lisible par l’homme."]}
for langue, fragments in phrases.items():
    print(langue, *[f"{i}. {phrase}" for i,phrase in enumerate(fragments,1)], sep="\n")

# Exécutez après avoir choisi la langue et le numéro (0 à 5) :
# langue, numero = "en-US", 0
# transcription_microphone(langue, duree=5)

## Exercice 2 — Pipeline d’évaluation OCR multilingue
Nous choisissons **anglais et français** (deux langues autorisées). Vous pouvez utiliser des images de corpus ou créer vos propres images. Pour un calcul WER/CER valide, renseignez une transcription de référence exacte de chaque image.

### 1 — Création d’un petit corpus de démonstration
Les images synthétiques servent à tester le pipeline ; pour un rapport expérimental, remplacez-les ou complétez-les par un corpus représentatif. Les références ci-dessous sont connues car les images sont générées à partir de ces textes.

In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter
corpus = {"en": "Natural language processing converts speech into text.",
          "fr": "La reconnaissance vocale convertit la parole en texte."}
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 27) if Path(font_path).exists() else ImageFont.load_default()
rows = []
for lang, reference in corpus.items():
    img = Image.new("RGB", (1000, 130), "white")
    draw = ImageDraw.Draw(img)
    draw.text((24, 40), reference, fill="black", font=font)
    chemin = DATA / f"ocr_{lang}.png"
    img.save(chemin)
    rows.append({"langue": lang, "image": str(chemin), "reference": reference})
df_corpus = pd.DataFrame(rows)
display(df_corpus)

### 2 — Deux moteurs OCR : Tesseract et EasyOCR
**Prérequis :** l’exécutable Tesseract et les données de langue doivent être installés séparément. EasyOCR télécharge ses modèles à la première exécution. Le temps mesuré inclut l’inférence mais pas l’initialisation des modèles EasyOCR.

In [ ]:
import pytesseract, easyocr
from PIL import Image
readers = {"en": easyocr.Reader(["en"], gpu=False),
           "fr": easyocr.Reader(["fr", "en"], gpu=False)}

def extraire_tesseract(image, langue):
    return pytesseract.image_to_string(Image.open(image), lang={"en":"eng", "fr":"fra"}[lang]).strip()

def extraire_easyocr(image, langue):
    return " ".join(readers[lang].readtext(image, detail=0, paragraph=True)).strip()

resultats = []
for item in rows:
    for moteur, fonction in [("Tesseract", extraire_tesseract), ("EasyOCR", extraire_easyocr)]:
        start = time.perf_counter()
        try:
            texte = fonction(item["image"], item["langue"])
            erreur = ""
        except Exception as exc:
            texte = ""
            erreur = str(exc)
        duree = time.perf_counter() - start
        resultats.append({**item, "moteur": moteur, "texte_brut": texte,
                          "temps_s": round(duree, 3), "erreur_execution": erreur})
df_ocr = pd.DataFrame(resultats)
display(df_ocr[["langue", "moteur", "texte_brut", "temps_s", "erreur_execution"]])

### 3 — Comparaison WER / CER et erreurs visibles
**WER** : proportion d’erreurs au niveau des mots. **CER** : proportion d’erreurs au niveau des caractères. Une valeur de 0 est idéale. Les deux métriques peuvent dépasser 1 dans certains cas. Une note qualitative de lisibilité sur 5 doit être attribuée manuellement.

In [ ]:
from jiwer import wer, cer
def calculer_metriques(df):
    df = df.copy()
    df["WER"] = df.apply(lambda x: wer(x["reference"], x["texte_brut"]) if not x["erreur_execution"] else float("nan"), axis=1)
    df["CER"] = df.apply(lambda x: cer(x["reference"], x["texte_brut"]) if not x["erreur_execution"] else float("nan"), axis=1)
    df["lisibilite_sur_5"] = None  # À remplir après inspection humaine
    df["fautes_visibles"] = None  # À remplir après inspection humaine
    return df
df_ocr = calculer_metriques(df_ocr)
display(df_ocr[["langue", "moteur", "WER", "CER", "temps_s", "fautes_visibles", "lisibilite_sur_5"]])

### 4 — Correction grammaticale avec LanguageTool
Le paquet `language_tool_python` peut nécessiter Java pour le serveur local. La correction est appliquée uniquement aux sorties OCR réussies. Une correction grammaticale peut modifier un texte exact : vérifier les métriques avant/après.

In [ ]:
import language_tool_python
tools_lang = {}
def corriger_texte(texte, langue):
    code_langue = {"en": "en-US", "fr": "fr-FR"}[langue]
    if code_langue not in tools_lang:
        tools_lang[code_langue] = language_tool_python.LanguageTool(code_langue)  # Java requis pour le serveur local
    outil = tools_lang[code_langue]
    matches = outil.check(texte)
    corrige = language_tool_python.utils.correct(texte, matches)
    return corrige, len(matches)

df_ocr["texte_corrige"] = ""
df_ocr["suggestions_correction"] = 0
df_ocr["erreur_correction"] = ""
for idx, row in df_ocr.iterrows():
    if row["erreur_execution"]:
        continue
    try:
        corrige, n = corriger_texte(row["texte_brut"], row["langue"])
        df_ocr.at[idx, "texte_corrige"] = corrige
        df_ocr.at[idx, "suggestions_correction"] = n
    except Exception as e:
        df_ocr.at[idx, "texte_corrige"] = row["texte_brut"]
        df_ocr.at[idx, "erreur_correction"] = str(e)
display(df_ocr[["langue", "moteur", "texte_brut", "texte_corrige", "suggestions_correction", "erreur_correction"]])

### 5 — Impact de la correction sur WER, CER et une tâche NLP
Comme exemple simple de tâche NLP, on utilise une **classification par mots-clés** (speech vs autre). C’est une démonstration et non un modèle de classification entraîné. Les suggestions de LanguageTool ne sont pas nécessairement toutes des erreurs effectivement corrigées.

In [ ]:
def classer_texte(texte):
    t = texte.casefold()
    return "reconnaissance vocale" if any(m in t for m in ["speech", "parole", "vocale"]) else "autre"

df_ocr["WER_apres"] = df_ocr.apply(lambda x: wer(x["reference"], x["texte_corrige"]) if not x["erreur_execution"] else float("nan"), axis=1)
df_ocr["CER_apres"] = df_ocr.apply(lambda x: cer(x["reference"], x["texte_corrige"]) if not x["erreur_execution"] else float("nan"), axis=1)
df_ocr["classe_avant"] = df_ocr["texte_brut"].apply(classer_texte)
df_ocr["classe_apres"] = df_ocr["texte_corrige"].apply(classer_texte)
df_ocr["amelioration_WER"] = df_ocr["WER"] - df_ocr["WER_apres"]
display(df_ocr[["langue", "moteur", "WER", "WER_apres", "CER", "CER_apres", "suggestions_correction", "classe_avant", "classe_apres"]])
df_ocr.to_csv("resultats_tp3_ocr.csv", index=False, encoding="utf-8-sig")
print("Résultats enregistrés dans resultats_tp3_ocr.csv")

### Conclusion à compléter après exécution
- Quel moteur produit le WER/CER le plus faible dans chaque langue ?
- Quel moteur est le plus rapide ?
- Quels caractères ou mots sont mal reconnus ? Quelle note de lisibilité (sur 5) attribuer ?
- Combien de suggestions ont été effectivement appliquées ? La correction améliore-t-elle la fluidité ?
- La correction améliore-t-elle le WER/CER et modifie-t-elle la classification ?

**Attention :** les résultats dépendent des images, des versions des moteurs et des modèles. Aucune mesure n’est inventée dans ce notebook.